# Stage 5 — Coding Attention

Build up to the attention block used in a GPT, step by step:

1. **Simplified self-attention** — dot products + softmax, no learned weights.
2. **Self-attention with trainable Q/K/V**.
3. **Causal (masked) attention** — no peeking at future tokens.
4. **Multi-head attention** — multiple parallel heads, concatenated.
5. **Wire it into a forward pass** with token + positional embeddings.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

device: mps


## 1) Simplified self-attention (no learned weights)

For each query token `i`, compute attention weights as softmax of dot products with every other token, then take the weighted sum.

`context_i = sum_j softmax(x_i · x_j) * x_j`

In [2]:
# 6 tokens, each embedded in 3 dims
inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # 'Your'
    [0.55, 0.87, 0.66],  # 'journey'
    [0.57, 0.85, 0.64],  # 'starts'
    [0.22, 0.58, 0.33],  # 'with'
    [0.77, 0.25, 0.10],  # 'one'
    [0.05, 0.80, 0.55],  # 'step'
])

attn_scores  = inputs @ inputs.T              # (T, T)
attn_weights = torch.softmax(attn_scores, dim=-1)
context_vecs = attn_weights @ inputs          # (T, D)

print('attn_weights row sums (should be 1):', attn_weights.sum(dim=-1))
print('context_vecs:\n', context_vecs)

attn_weights row sums (should be 1): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
context_vecs:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 2) Self-attention with trainable Q, K, V

Add three learned projections. Scale scores by `sqrt(d_k)` (Vaswani et al.) so softmax doesn't saturate as `d_k` grows.

In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        scores  = q @ k.transpose(-2, -1) / (k.size(-1) ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        return weights @ v

sa = SelfAttention(d_in=3, d_out=2)
print(sa(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


## 3) Causal (masked) self-attention

Mask out future positions so a token at position `t` can only attend to positions `<= t`. Also add dropout on the weights (regularization, as in GPT).

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_len, dropout=0.0, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # Buffer: not a parameter, but moves with the module .to(device).
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_len, context_len), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = q @ k.transpose(-2, -1)
        scores = scores.masked_fill(self.mask[:T, :T], float('-inf'))
        weights = torch.softmax(scores / (k.size(-1) ** 0.5), dim=-1)
        weights = self.dropout(weights)
        return weights @ v

batch = inputs.unsqueeze(0)  # (1, 6, 3)
csa = CausalSelfAttention(d_in=3, d_out=2, context_len=6, dropout=0.0)
out = csa(batch)
print('out shape:', out.shape)
print(out)

out shape: torch.Size([1, 6, 2])
tensor([[[0.4772, 0.1063],
         [0.5891, 0.3257],
         [0.6202, 0.3860],
         [0.5478, 0.3589],
         [0.5321, 0.3428],
         [0.5077, 0.3493]]], grad_fn=<UnsafeViewBackward0>)


## 4) Multi-head attention

Run `h` causal attention heads in parallel on splits of the embedding, then concat and project. This is the efficient single-matmul version used in real GPTs.

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_len, num_heads, dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, 'd_out must be divisible by num_heads'
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads
        self.d_out     = d_out
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_len, context_len), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape
        # (B, T, d_out) → (B, num_heads, T, head_dim)
        q = self.W_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores.masked_fill(self.mask[:T, :T], float('-inf'))
        weights = torch.softmax(scores / (self.head_dim ** 0.5), dim=-1)
        weights = self.dropout(weights)

        ctx = (weights @ v).transpose(1, 2).contiguous().view(B, T, self.d_out)
        return self.out_proj(ctx)

mha = MultiHeadAttention(d_in=3, d_out=4, context_len=6, num_heads=2, dropout=0.0)
print(mha(batch).shape)  # → (1, 6, 4)

torch.Size([1, 6, 4])


## 5) Wire it to embeddings (real data path)

What the real LLM forward pass looks like up to one attention block:
1. Token IDs → token embeddings.
2. Add positional embeddings.
3. Run through multi-head causal attention.

We use the tokenized Mahabharata from Stage 4.

In [6]:
import numpy as np
from pathlib import Path

ids_path = Path('data/processed/train.bin')
if ids_path.exists():
    train_ids = np.fromfile(ids_path, dtype=np.uint16).astype(np.int64)
    print(f'loaded {len(train_ids):,} train tokens')
else:
    # Fallback to a tiny synthetic batch so this notebook runs standalone.
    train_ids = np.random.randint(0, 50257, size=10_000)
    print('train.bin missing — using random ids for demo')

VOCAB_SIZE  = 50257   # GPT-2 BPE
CONTEXT_LEN = 128
EMB_DIM     = 128
NUM_HEADS   = 4
BATCH       = 4

# Sample a random batch.
starts = np.random.randint(0, len(train_ids) - CONTEXT_LEN - 1, size=BATCH)
xb = torch.tensor(np.stack([train_ids[i:i+CONTEXT_LEN] for i in starts]), dtype=torch.long)
yb = torch.tensor(np.stack([train_ids[i+1:i+1+CONTEXT_LEN] for i in starts]), dtype=torch.long)
print('xb:', xb.shape, ' yb:', yb.shape)

loaded 362,720 train tokens
xb: torch.Size([4, 128])  yb: torch.Size([4, 128])


In [7]:
class EmbedAndAttend(nn.Module):
    """Token+positional embeddings → one multi-head causal attention block."""
    def __init__(self, vocab_size, emb_dim, context_len, num_heads, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(context_len, emb_dim)
        self.drop    = nn.Dropout(dropout)
        self.attn    = MultiHeadAttention(
            d_in=emb_dim, d_out=emb_dim,
            context_len=context_len, num_heads=num_heads, dropout=dropout,
        )

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)   # (B, T, E)
        x = self.drop(x)
        return self.attn(x)                          # (B, T, E)

model = EmbedAndAttend(VOCAB_SIZE, EMB_DIM, CONTEXT_LEN, NUM_HEADS).to(device)
out = model(xb.to(device))
print('forward out:', out.shape)
n_params = sum(p.numel() for p in model.parameters())
print(f'trainable params: {n_params:,}')

forward out: torch.Size([4, 128, 128])
trainable params: 6,514,944


### What's left for a full GPT

From here, the next stages would be:

- **Transformer block** = `LayerNorm → MHA → residual → LayerNorm → FFN(GELU) → residual`
- **GPT model** = embed → N transformer blocks → final LayerNorm → linear head → vocab logits
- **Training loop** with `cross_entropy(logits.view(-1, V), targets.view(-1))` and AdamW
- **Generation** with temperature + top-k sampling

Drop a follow-up when you want stages 6–8.